# אדם בתוך הלולאה: שערי טרום-פעולה, דירוג סיכונים, ורישום ביקורת

קובץ ה-README של השיעור הזה מציג את מושג אדם בתוך הלולאה עם קטע קצר שמבקש מהמשתמש `APPROVE` או `REJECT` לאחר שהסוכן כבר הפיק תגובה. התבנית הזו היא נקודת התחלה טובה, אך יישומים מבוססי HITL בייצור בדרך כלל צריכים שלושה פריטים נוספים:

1. **שער טרום-פעולה** שמופעל **לפני** שהסוכן מבצע שלב מסוכן, כדי לשמור על עלויות, אי-הפיכות ושהייה בזמנים תחת שליטה.
2. **דירוג סיכונים**, כך שפעולות בסיכון נמוך מבוצעות אוטומטית, פעולות בסיכון בינוני מאושרות בקבוצות, ורק פעולות בסיכון גבוה מעכבות על אדם.
3. **יומן ביקורת ולולאת תיקונים**, כך שכל החלטת שער מתועדת כ-JSONL, ודחייה מחזירה את הסוכן עם סיבה מובנית במקום רק להדפיס `Revising...`.

מחברת זו בונה כל אחד מהרכיבים האלה על בסיס אותם פרימיטיבים כמו `06-system-message-framework.ipynb`. היא רצה מקצה לקצה ב-`DEMO_MODE = True` (לא נדרש קלט אינטראקטיבי) או עם קלט ממשי של `input()` כאשר `DEMO_MODE = False`. הערה: במצב DEMO_MODE, הניסיון החוזר של המטרה השלישית מתוזמר כך שמכניקת הלולאה נראית מקצה לקצה. סיווג מבוסס תיקונים אמיתי דורש `DEMO_MODE = False` ומפעיל.

**מחוץ להיקף (מטופל בשיעורים אחרים):** אימות ושליטה בגישה (איום 2 של שיעור 06 README), תיווך קריאות כלים (מבט מעמיק ב- Lesson 14 MAF), דפוסי דיון של סוכנים מרובים.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## תבנית 1: שער לפני פעולה

הקטע HITL ב-README קורא לסוכן קודם, ואז מבקש מהמשתמש לאשר את הפלט. זהו זרם **אחרי הפעולה**. הסוכן כבר ביצע את הפעולה, אז עלות הקריאה ל-LLM כבר שולמה, וכל תופעת לוואי (שליחת מייל, שורה שנכתבה במסד נתונים, תגובה שפורסמה) כבר התרחשה.

זרם **לפני הפעולה** מוסיף את השער לפני שהסוכן מריץ את הצעד המסוכן. הסוכן מציע את הפעולה, השער מחליט האם לבצע אותה, ורק באישור מתרחשת תופעת הלוואי.

| היבט | אישור אחרי פעולה (קטע README) | שער לפני פעולה (פנקס זה) |
|---|---|---|
| מתי רץ האישור? | אחרי שהסוכן הפיק פלט | לפני ביצוע כל תופעת לוואי |
| עלות LLM במקרה של דחייה | כבר שולמה | משולם רק עבור ההצעה, לא עבור הפעולה |
| תופעות לוואי בלתי הפיכות | אפשריות (הפעולה כבר התרחשה) | נמנעות |
| בהירות ביקורת | האישור הוא משפט print | האישור הוא רשומת JSON עם חותמת זמן, פעולה, סיבה |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## דפוס 2: דירוג סיכון

לא כל פעולה דורשת אישור אנושי. חיפוש לקריאה בלבד מול API ציבורי הוא שונה ממשלוח דואר אלקטרוני ללקוח. טיפול בשניהם באותה צורה מבזבז את תשומת הלב של המפעיל ומאט את הסוכן.

מודל פשוט בעל 3 רמות:

| רמה | דוגמאות | זרימת אישור |
|---|---|---|
| `נמוכה` (לקריאה בלבד) | חיפוש במאגר ידע, בדיקת אפשרויות טיסה, משיכת דף אינטרנט ציבורי | ביצוע אוטומטי, נרשם לביקורת |
| `בינונית` (שינוי זול) | אחסון תוצאה בזיכרון מטמון, הגדלת מונה, תזמון תזכורת | ביצוע אוטומטי, אך סקירה יומית מצורפת |
| `גבוהה` (פונות כלפי חוץ או בלתי הפיכים) | שליחת דואר אלקטרוני, חיוב כרטיס, פרסום בערוץ ציבורי | עצירה עד לאישור אנושי |

זוהי דרגת סיכון אחת. במערכות ייצור לרוב משתמשים בדרגות עוד יותר מפורטות (למשל, רמות הרשאה ב-AWS IAM, דרגות גישה מבוססות תפקיד). גרסת ה-3 רמות כאן היא הגרסה הקטנה ביותר השימושית לסוכן המשלב פעולות לקריאה בלבד לצד פעולות עם תופעות לוואי.

המסווג למטה משתמש בהיוריסטיקות ממילות מפתח כדי שההדגמה תישאר דטרמיניסטית וזולה. במערכת ייצור תחליף זאת במסווג שלמד או במנוע מדיניות.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## דפוס 3: יומן ביקורת ולולאת סקירה מחדש

`print("Response approved.")` איננה יומן ביקורת. עבור אמון, כל החלטת שער צריכה להירשם כאירוע מובנה שניתן אחר כך לשאול, להשמיע מחדש, או לצרף לסקירת אירוע.

שני חלקים:

1. **JSONL לצירוף בלבד.** שורה אחת לכל החלטה, עם חותמת זמן, פעולה, שכבה, החלטה, סיבה. קל לחפש, קל לשלוח למאגר יומנים אמיתי מאוחר יותר.
2. **לולאת סקירה מחדש בדחייה.** כאשר השער מחזיר `deny`, הסוכן מפעיל מחדש את עצמו עם סיבת הדחייה בהקשר, כך שההצעה הבאה תוכל להימנע מהבעיה.


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## משאבים נוספים

מספר פרויקטים ציבוריים נוספים מממשים וריאציות של דפוסי HITL אלה. השוו גישות כדי למצוא מה מתאים לערמה שלכם:

- **LangChain** עטיפות כלים עם אדם בלולאה ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): עטיפות כלים להוספה שמפסיקות ביצוע לקלט אנושי.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ ארגן זאת מחדש): משתמש בתפקיד סוכן מיוחד לייצוג האדם בשיחות מרובי-סוכנים.
- **Microsoft Agent Framework (MAF)** תיווך קריאת פונקציות ([docs](https://learn.microsoft.com/agent-framework/)): תיווך שרץ סביב כל קריאה לכלי/פונקציה, מתאים ללוגיקה של סינון וזרמי אישורים.

כל פרויקט מטפל בשלוש תת-הדפוסים בצורה שונה: LangChain עוטף אותם ככלים, AutoGen משתמש בתפקיד סוכן, ו-Microsoft Agent Framework משתמש בתיווך קריאת פונקציות. קראו מימוש אחד או שניים מקצה לקצה לפני שתבחרו עיצוב לסוכן שלכם.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
